# Stage 06-C: Full Architecture Comparison

Aggregates results from all fusion experiments and produces a unified analysis.
Missing results are skipped gracefully — run whichever subset of notebooks you have.

## Notebooks Compared

| ID | Notebook | Fusion Strategy | Cache |
|---|---|---|---|
| 05a | `05a_mil_attention_fusion` | MIL attention over images (no text conditioning) | stage05a |
| 05b | `05b_asymmetric_gated_fusion` | Asymmetric gate + text residual (feature level) | stage05b |
| 05d | `05d_misinformation_aware_mil` | MIL + COOLANT fake_prob signal per pair | stage05d |
| 06a | `06a_cross_attention_fusion` | Text-conditioned cross-attention over image pairs | stage05a |
| 06b | `06b_late_decision_fusion` | Independent branches + meta-learner (decision level) | stage05b |

## Outputs
- `training/stage06c_results/comparison_table.csv`
- `training/stage06c_results/comparison_bar_f1.png`
- `training/stage06c_results/comparison_per_class_f1.png`
- `training/stage06c_results/confusion_matrices.png`
- `training/stage06c_results/training_curves_all.png`


In [ ]:
# ─── Environment Setup (do not edit) ─────────────────────────────────────────
import os, sys
from pathlib import Path

def _detect_platform():
    try:
        import google.colab
        return 'colab', False
    except ImportError:
        pass
    if Path('/workspace').exists() and os.environ.get('VAST_CONTAINERLABEL'):
        return 'vastai', False
    if Path('/workspace').exists():
        return 'vastai', True
    if sys.platform == 'win32': return 'windows', False
    if sys.platform == 'darwin': return 'mac', False
    return None, True

PLATFORM, _uncertain = _detect_platform()
if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

PROJECT_ROOT = (
    Path('/content/drive/MyDrive/Thesis_Final/fake-news-detection') if PLATFORM == 'colab'
    else _nb_path.parents[1]
)
sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {
    'colab':   PROJECT_ROOT / '.env.colab',
    'vastai':  PROJECT_ROOT / '.env.vastai',
    'windows': PROJECT_ROOT / '.env.windows',
    'mac':     PROJECT_ROOT / '.env.mac',
}
_env_file = _env_map.get(PLATFORM, PROJECT_ROOT / '.env')
if not _env_file.exists():
    _env_file = PROJECT_ROOT / '.env'

from dotenv import load_dotenv
load_dotenv(_env_file, override=True)
from src.utils.env_utils import get_data_root
DATA_ROOT = get_data_root()

print(f'Platform : {PLATFORM}  DATA_ROOT: {DATA_ROOT}')

## Step 1: Configuration


In [ ]:
import os, json, importlib
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == 'pipeline' else Path.cwd()
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / '.env', override=False)
except ImportError:
    pass
DATA_ROOT = Path(os.environ['DATA_ROOT']) if os.environ.get('DATA_ROOT') else PROJECT_ROOT

# All experiment result directories to scan
EXPERIMENTS = [
    {
        'id':       '05a',
        'label':    '05a MIL Attention',
        'strategy': 'MIL self-attention\n(image-only query)',
        'color':    '#6366F1',
        'results_dir': DATA_ROOT / 'training' / 'stage05a_results',
    },
    {
        'id':       '05b',
        'label':    '05b Asym Gate',
        'strategy': 'Asymmetric gate\n+ text residual',
        'color':    '#F59E0B',
        'results_dir': DATA_ROOT / 'training' / 'stage05b_results',
    },
    {
        'id':       '05d',
        'label':    '05d Misinfo MIL',
        'strategy': 'MIL + COOLANT\nfake_prob signal',
        'color':    '#EF4444',
        'results_dir': DATA_ROOT / 'training' / 'stage05d_results',
    },
    {
        'id':       '06a',
        'label':    '06a Cross-Attn',
        'strategy': 'Cross-attention\n(text-conditioned)',
        'color':    '#0EA5E9',
        'results_dir': DATA_ROOT / 'training' / 'stage06a_results',
    },
    {
        'id':       '06b',
        'label':    '06b Decision Fusion',
        'strategy': 'Independent branches\n+ meta-learner',
        'color':    '#10B981',
        'results_dir': DATA_ROOT / 'training' / 'stage06b_results',
    },
]

OUTPUT_DIR = DATA_ROOT / 'training' / 'stage06c_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

## Step 2: Load Results


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

def load_latest_results(results_dir):
    """Load the most recent *_results.json from a results directory."""
    d = Path(results_dir)
    if not d.exists():
        return None
    cands = sorted(d.glob('*_results.json'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not cands:
        return None
    with open(cands[0]) as f:
        r = json.load(f)
    r['_source_file'] = str(cands[0].name)
    return r

loaded = []
for exp in EXPERIMENTS:
    r = load_latest_results(exp['results_dir'])
    if r is None:
        print(f'  [SKIP] {exp["label"]:25s} — results not found ({exp["results_dir"]})')
    else:
        exp['results'] = r
        loaded.append(exp)
        tm = r.get('test_metrics', {})
        print(f'  [OK]   {exp["label"]:25s} — macro_f1={tm.get("test_macro_f1"):.4f}  '
              f'acc={tm.get("test_accuracy"):.4f}  file={r["_source_file"]}')

print(f'\nLoaded {len(loaded)}/{len(EXPERIMENTS)} experiments.')
if not loaded:
    print('No results found. Run the training notebooks first.')

## Step 3: Comparison Table


In [ ]:
rows = []
for exp in loaded:
    tm = exp['results'].get('test_metrics', {})
    vm = exp['results'].get('val_metrics', {})
    rows.append({
        'ID':               exp['id'],
        'Model':            exp['label'],
        'Strategy':         exp['strategy'].replace('\n', ' / '),
        'Best Epoch':       exp['results'].get('best_epoch', '?'),
        'Val Macro-F1':     round(exp['results'].get('best_val_macro_f1', 0), 4),
        'Test Macro-F1':    round(tm.get('test_macro_f1', 0), 4),
        'Test Accuracy':    round(tm.get('test_accuracy', 0), 4),
        'F1-Real':          round(tm.get('test_f1_real', 0), 4),
        'F1-Fake':          round(tm.get('test_f1_fake', 0), 4),
        'F1-NEI':           round(tm.get('test_f1_nei', 0), 4),
    })

if rows:
    df = pd.DataFrame(rows).sort_values('Test Macro-F1', ascending=False).reset_index(drop=True)
    df.index += 1  # 1-indexed rank
    df.index.name = 'Rank'

    # Highlight best per column
    best_f1  = df['Test Macro-F1'].max()
    best_acc = df['Test Accuracy'].max()

    print('\nFull Comparison Table (sorted by Test Macro-F1):')
    print('=' * 100)
    print(df.to_string())
    print('=' * 100)

    csv_path = OUTPUT_DIR / 'comparison_table.csv'
    df.to_csv(csv_path)
    print(f'\nTable saved: {csv_path}')

    # Rank 1 summary
    best_row = df.iloc[0]
    print(f'\n\u2605  Best model: [{best_row["Model"]}]')
    print(f'   Test Macro-F1={best_row["Test Macro-F1"]:.4f}  '
          f'Accuracy={best_row["Test Accuracy"]:.4f}  '
          f'F1-Fake={best_row["F1-Fake"]:.4f}')

## Step 4: Macro-F1 and Per-Class F1 Bar Charts


In [ ]:
if loaded:
    labels  = [exp['label'] for exp in loaded]
    colors  = [exp['color'] for exp in loaded]
    macro   = [exp['results'].get('test_metrics', {}).get('test_macro_f1', 0) for exp in loaded]
    f1_real = [exp['results'].get('test_metrics', {}).get('test_f1_real', 0)  for exp in loaded]
    f1_fake = [exp['results'].get('test_metrics', {}).get('test_f1_fake', 0)  for exp in loaded]
    f1_nei  = [exp['results'].get('test_metrics', {}).get('test_f1_nei',  0)  for exp in loaded]

    # ── Macro F1 bar ─────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(max(8, len(loaded) * 1.8), 5))
    bars = ax.bar(labels, macro, color=colors, edgecolor='white', linewidth=1.2, zorder=2)
    for bar, val in zip(bars, macro):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.axhline(max(macro), color='gray', linestyle='--', linewidth=0.8, alpha=0.6, label=f'best={max(macro):.4f}')
    ax.set_ylabel('Test Macro-F1'); ax.set_title('Test Macro-F1 — All Fusion Strategies')
    ax.set_ylim(max(0.0, min(macro) - 0.05), min(1.0, max(macro) + 0.05))
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3, zorder=0)
    plt.xticks(rotation=15, ha='right', fontsize=9)
    plt.tight_layout()
    p = OUTPUT_DIR / 'comparison_bar_f1.png'
    plt.savefig(p, dpi=150); plt.show(); plt.close()
    print(f'Saved: {p}')

    # ── Per-class F1 grouped bar ──────────────────────────────────────────────
    x     = np.arange(len(loaded))
    width = 0.25
    fig, ax = plt.subplots(figsize=(max(10, len(loaded) * 2.2), 5))
    b1 = ax.bar(x - width, f1_real, width, label='F1-Real (class 0)', color='#22C55E', alpha=0.85, edgecolor='white')
    b2 = ax.bar(x,         f1_fake, width, label='F1-Fake (class 1)', color='#EF4444', alpha=0.85, edgecolor='white')
    b3 = ax.bar(x + width, f1_nei,  width, label='F1-NEI  (class 2)', color='#A78BFA', alpha=0.85, edgecolor='white')
    for bars in (b1, b2, b3):
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
    ax.set_ylabel('F1 Score'); ax.set_title('Per-Class F1 — All Fusion Strategies')
    ax.set_ylim(0, min(1.05, max(max(f1_real), max(f1_fake), max(f1_nei)) + 0.08))
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    p = OUTPUT_DIR / 'comparison_per_class_f1.png'
    plt.savefig(p, dpi=150); plt.show(); plt.close()
    print(f'Saved: {p}')

## Step 5: Confusion Matrices (Side by Side)


In [ ]:
cms = [(exp['label'], exp['results']['test_metrics'].get('confusion_matrix'))
       for exp in loaded
       if exp['results'].get('test_metrics', {}).get('confusion_matrix')]

if cms:
    n = len(cms)
    fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 4.5))
    if n == 1: axes = [axes]
    cmaps = ['Blues', 'Oranges', 'Reds', 'PuBu', 'Greens']
    for idx, ((label, cm), ax, cmap) in enumerate(zip(cms, axes, cmaps * 2)):
        cm_arr = np.array(cm)
        sns.heatmap(cm_arr, annot=True, fmt='d', cmap=cmap,
                    xticklabels=['Real', 'Fake', 'NEI'],
                    yticklabels=['Real', 'Fake', 'NEI'],
                    ax=ax, cbar=False)
        ax.set_title(label, fontsize=9)
        ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.suptitle('Test Confusion Matrices — All Models', fontsize=12, y=1.02)
    plt.tight_layout()
    p = OUTPUT_DIR / 'confusion_matrices.png'
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show(); plt.close()
    print(f'Saved: {p}')
else:
    print('No confusion matrices found in results.')

## Step 6: Validation Macro-F1 Training Curves


In [ ]:
histories = [(exp['label'], exp['color'], exp['results'].get('training_history', []))
             for exp in loaded if exp['results'].get('training_history')]

if histories:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for label, color, hist in histories:
        epochs  = [r['epoch'] for r in hist]
        val_f1  = [r.get('val_macro_f1', 0) for r in hist]
        val_acc = [r.get('val_accuracy',  0) for r in hist]
        axes[0].plot(epochs, val_f1,  label=label, color=color, linewidth=2)
        axes[1].plot(epochs, val_acc, label=label, color=color, linewidth=2)

    for ax, title in zip(axes, ['Val Macro-F1 per Epoch', 'Val Accuracy per Epoch']):
        ax.set_xlabel('Epoch'); ax.set_ylabel(title.split(' ')[1])
        ax.set_title(title); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    plt.suptitle('Training Curves — All Fusion Strategies', fontsize=12)
    plt.tight_layout()
    p = OUTPUT_DIR / 'training_curves_all.png'
    plt.savefig(p, dpi=150); plt.show(); plt.close()
    print(f'Saved: {p}')
else:
    print('No training history found in results.')

## Step 7: Architecture Comparison Summary & Verdict


In [ ]:
if not loaded:
    print('No results loaded. Run training notebooks first.')
else:
    # Sort by test macro-F1
    ranked = sorted(loaded, key=lambda e: e['results'].get('test_metrics', {}).get('test_macro_f1', 0), reverse=True)

    print('\n' + '=' * 70)
    print('FUSION STRATEGY COMPARISON — FINAL VERDICT')
    print('=' * 70)

    arch_notes = {
        '05a': 'Image self-attention (MIL). Text not involved in attention scoring.',
        '05b': 'Feature-level gate. Text and image mixed at hidden dim.',
        '05d': 'MIL + COOLANT fake_prob. Exploits per-pair mismatch signal.',
        '06a': 'Text-conditioned cross-attention. Claim selects which images matter.',
        '06b': 'Decision-level stacking. Branches independent; meta-learner fuses.',
    }

    for rank, exp in enumerate(ranked, 1):
        tm = exp['results'].get('test_metrics', {})
        f1   = tm.get('test_macro_f1', 0)
        acc  = tm.get('test_accuracy', 0)
        fake = tm.get('test_f1_fake', 0)
        ep   = exp['results'].get('best_epoch', '?')
        marker = '\u2605 ' if rank == 1 else f'{rank}.  '
        print(f'\n{marker}[{exp["label"]}]')
        print(f'   Strategy : {arch_notes.get(exp["id"], "?")}')
        print(f'   Test F1   : {f1:.4f}  Acc: {acc:.4f}  F1-Fake: {fake:.4f}  Best Epoch: {ep}')

    best = ranked[0]
    print('\n' + '-' * 70)
    print(f'Winner: [{best["label"]}]')
    print(f'  Test Macro-F1 = {best["results"]["test_metrics"].get("test_macro_f1", 0):.4f}')

    # Performance delta from runner-up
    if len(ranked) >= 2:
        delta = (best['results']['test_metrics'].get('test_macro_f1', 0)
                 - ranked[1]['results']['test_metrics'].get('test_macro_f1', 0))
        print(f'  Margin over #{2} ({ranked[1]["label"]}): {delta:+.4f}')

    # NEI analysis: hardest class across models
    print('\n--- NEI (hard class) analysis ---')
    for exp in ranked:
        nei = exp['results'].get('test_metrics', {}).get('test_f1_nei', 0)
        print(f'  {exp["label"]:28s}: F1-NEI={nei:.4f}')

    print('\n' + '=' * 70)
    print(f'Report generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

    # Save summary JSON
    summary = {
        'generated': datetime.now().isoformat(),
        'experiments_compared': len(loaded),
        'ranking': [
            {
                'rank': i+1,
                'id':    exp['id'],
                'label': exp['label'],
                'test_macro_f1': exp['results'].get('test_metrics', {}).get('test_macro_f1'),
                'test_accuracy': exp['results'].get('test_metrics', {}).get('test_accuracy'),
                'test_f1_fake':  exp['results'].get('test_metrics', {}).get('test_f1_fake'),
                'test_f1_nei':   exp['results'].get('test_metrics', {}).get('test_f1_nei'),
                'best_epoch':    exp['results'].get('best_epoch'),
            }
            for i, exp in enumerate(ranked)
        ],
    }
    sp = OUTPUT_DIR / 'comparison_summary.json'
    with open(sp, 'w') as f: json.dump(summary, f, indent=2)
    print(f'Summary JSON saved: {sp}')